# Alocação enfermeiro-quarto por turno (NRA)

## Programação Linear Inteira (PLI)

In [ ]:
import pulp
import pandas as pd
import json
import time
import os
import random
import copy

def solve_nra_problem(instance_dir):
    # start of time measurement
    start_time = time.perf_counter()

    # data load
    with open(f"{instance_dir}/instance_info.json", "r") as f:
        info = json.load(f)

    nurse_shifts = pd.read_csv(f"{instance_dir}/nurse_shifts.csv")
    room_shifts = pd.read_csv(f"{instance_dir}/occupied_room_shifts.csv")

    # extract penalties weights
    w_skill = info["weights"]["S2_room_nurse_skill"]
    w_work = info["weights"]["S4_nurse_excessive_workload"]

    # init the solver
    prob = pulp.LpProblem("NRA_Optimization", pulp.LpMinimize)

    # binary variables: 1 if the nurse is allocated to the room, otherwise 0
    x_vars = {}  # x[n, r, s]

    # integer variables: nurse workload
    e_vars = {}  # e[n, s]

    objective_terms = []

    # get the each turn
    shifts_unique = room_shifts["global_shift"].unique()

    # for each turn: 0, 1, ..., 41
    for s in shifts_unique:
        rooms_in_shift = room_shifts[room_shifts["global_shift"] == s]
        available_nurses = nurse_shifts[nurse_shifts["global_shift"] == s]

        if available_nurses.empty and not rooms_in_shift.empty:
            continue
        
        # for each nurse
        for _, nurse in available_nurses.iterrows():
            n_id = nurse["nurse_id"]
            n_skill = nurse["skill_level"]
            n_capacity = nurse["max_load"]  

            e_vars[(n_id, s)] = pulp.LpVariable(
                f"e_{n_id}_{s}", lowBound=0, cat=pulp.LpInteger
            )

            # add the penalty weight
            objective_terms.append(w_work * e_vars[(n_id, s)])

            workload_sum = 0

            for _, room in rooms_in_shift.iterrows():
                r_id = room["room_id"]
                r_req_skill = room["max_skill_required"]
                r_workload = room["total_room_workload"]

                # binary variable
                x_vars[(n_id, r_id, s)] = pulp.LpVariable(
                    f"x_{n_id}_{r_id}_{s}", cat=pulp.LpBinary
                )

                # skill deficit
                deficit = max(0, r_req_skill - n_skill)

                # add the deficit
                if deficit > 0:
                    objective_terms.append(w_skill * deficit * x_vars[(n_id, r_id, s)])

                # total workload
                workload_sum += r_workload * x_vars[(n_id, r_id, s)]

            # soft constraint: workload
            prob += (
                workload_sum - n_capacity <= e_vars[(n_id, s)],
                f"Workload_{n_id}_{s}",
            )

        # hard constraint: room coverage 
        for _, room in rooms_in_shift.iterrows():
            r_id = room["room_id"]
            prob += (
                pulp.lpSum(
                    [
                        x_vars[(n["nurse_id"], r_id, s)]
                        for _, n in available_nurses.iterrows()
                    ]
                )
                == 1, # c_1 + c_2 + ... + c_n = 1
                f"Coverage_{r_id}_{s}",
            )

    # calculate all penalties
    prob += pulp.lpSum(objective_terms), "Total_Penalties"

    # solve with CBC
    prob.solve(pulp.PULP_CBC_CMD(msg=True))

    # end time measurement
    end_time = time.perf_counter()
    execution_time = end_time - start_time

    # results
    penalidade_total = pulp.value(prob.objective)
    
    print("\n" + "="*50)
    print("RESULTADOS DO MÉTODO EXATO (PLI - Solver CBC)")
    print("="*50)
    print(f"Status da Resolução                 : {pulp.LpStatus[prob.status]}")
    print(f"Função Objetivo (Penalidade Total)  : {penalidade_total}")
    print(f"Tempo de Processamento              : {execution_time:.2f} segundos")
    print("="*50)

    # save in a json
    ARQUIVO_RESULTADOS = 'resultados_comparativos.json'
    
    if os.path.exists(ARQUIVO_RESULTADOS):
        with open(ARQUIVO_RESULTADOS, 'r', encoding='utf-8') as f:
            dados_json = json.load(f)
    else:
        dados_json = {
            "PLI": {},
            "GA": {
                "penalidade_melhor": 0.0,
                "penalidade_media": 0.0,
                "tempo_medio_segundos": 0.0
            }
        }
    
    # update the PLI keys with the exact results
    dados_json["PLI"]["penalidade_total"] = float(penalidade_total) if penalidade_total is not None else 0.0
    dados_json["PLI"]["tempo_segundos"] = float(execution_time)

    # write back to the file
    with open(ARQUIVO_RESULTADOS, 'w', encoding='utf-8') as f:
        json.dump(dados_json, f, indent=4)

    print(f"-> Arquivo '{ARQUIVO_RESULTADOS}' atualizado com as métricas do PLI!\n")

    return prob

## Metaheurística: Algoritmo Genético

In [ ]:
class NRA_Environment:
    """
    Class to load and encapsulate the problem data for a specific shift.
    """

    def __init__(
        self, rooms_df: pd.DataFrame, nurses_df: pd.DataFrame, w_skill: int, w_work: int
    ):
        self.rooms = rooms_df.to_dict("records")
        self.nurses: dict[str, dict] = {
            n["nurse_id"]: n for n in nurses_df.to_dict("records")
        }
        self.nurse_ids = list(self.nurses.keys())
        self.w_skill: int = w_skill
        self.w_work: int = w_work

    def calculate_fitness(self, chromosome: list[str]) -> int:
        """
        Calculate the total penalty. Less is better.
        """
        penalty: int = 0
        workload_sum: dict[str, int] = {n_id: 0 for n_id in self.nurse_ids}

        # room (gene index) and nurse (gene value)
        for room_idx, nurse_id in enumerate(chromosome):
            room: dict = self.rooms[room_idx]
            nurse: dict = self.nurses[nurse_id]

            # hability deficit penalty
            deficit = max(0, room["max_skill_required"] - nurse["skill_level"])
            penalty += self.w_skill * deficit

            # Acumular carga de trabalho para este enfermeiro
            workload_sum[nurse_id] += room["total_room_workload"]

        # workload penalty
        for nurse_id, total_load in workload_sum.items():
            nurse: dict = self.nurses[nurse_id]
            excess: int = max(0, total_load - nurse["max_load"])
            penalty += self.w_work * excess

        return penalty


class GeneticAlgorithm:
    def __init__(
        self,
        env: NRA_Environment,
        pop_size: int = 100,
        mutation_rate: float = 0.1,
        generations: int = 200,
        tournament_size: int = 3,
    ):
        self.env: NRA_Environment = env
        self.pop_size: int = pop_size
        self.mutation_rate: float = mutation_rate
        self.generations: int = generations
        self.tournament_size: int = tournament_size
        self.num_genes: int = len(self.env.rooms)
        self.history_best: list[int | float] = (
            []
        )

    def create_individual(self) -> list[str]:
        """
        Create a valid cromossome
        """
        # Chromosomes are lists of strings that represent which nurse is in the room
        # the list index represents the room
        return [random.choice(self.env.nurse_ids) for _ in range(self.num_genes)]

    def crossover(self, parent1: list[str], parent2: list[str]) -> list[str]:
        """
        Randomly choose who will inherit the gene (nurse in a room).
        """
        child: list[str] = []
        for i in range(self.num_genes):
            if random.random() < 0.5:
                child.append(parent1[i])
            else:
                child.append(parent2[i])
        return child

    def mutate(self, individual: list[str]) -> list[str]:
        """
        Randomly choose a room and change the nurse to other avaiable nurse.
        """
        for i in range(self.num_genes):
            if random.random() < self.mutation_rate:
                individual[i] = random.choice(self.env.nurse_ids)
        return individual

    def tournament_selection(self, population, fitnesses) -> list[str]:
        """
        Select the best cromossome between 'k' randomly chosen.
        """
        # list with "k" penalties of random cromossomes
        selected_indices: list[int] = random.sample(
            range(self.pop_size), self.tournament_size
        )
        best_idx: int = min(selected_indices, key=lambda idx: fitnesses[idx])

        # returns the best cromossome
        return population[best_idx]

    def run(self):
        # init the population
        population: list[list[str]] = [
            self.create_individual() for _ in range(self.pop_size)
        ]

        best_overall_fitness = float("inf")
        best_overall_individual = None

        for gen in range(self.generations):
            fitnesses: list[int] = [
                self.env.calculate_fitness(ind) for ind in population
            ]

            # store the best iteration
            current_best_fit: int = min(fitnesses)
            if current_best_fit < best_overall_fitness:
                best_overall_fitness = current_best_fit
                best_overall_individual = copy.deepcopy(
                    population[fitnesses.index(current_best_fit)]
                )

            # add the best population
            self.history_best.append(best_overall_fitness)

            # keep the absolute best indiviual
            new_population: list[list[str]] = [best_overall_individual]  # type: ignore

            # make a new population
            while len(new_population) < self.pop_size:
                # get the best element with tournament to do crossover and mutation
                parent_1: list[str] = self.tournament_selection(population, fitnesses)
                parent_2: list[str] = self.tournament_selection(population, fitnesses)

                # cross the bests
                child: list[str] = self.crossover(parent_1, parent_2)

                # mutation
                child = self.mutate(child)

                # add the new cromossome in the population
                new_population.append(child)

            # update the population
            population = new_population

        return best_overall_individual, best_overall_fitness


def solve_nra_metaheuristic(instance_dir: str, repetitions: int = 5):
    # data load
    with open(f"{instance_dir}/instance_info.json", "r") as f:
        info = json.load(f)

    nurse_shifts: pd.DataFrame = pd.read_csv(f"{instance_dir}/nurse_shifts.csv")
    room_shifts: pd.DataFrame = pd.read_csv(f"{instance_dir}/occupied_room_shifts.csv")

    # weights
    w_skill: int = info["weights"]["S2_room_nurse_skill"]
    w_work: int = info["weights"]["S4_nurse_excessive_workload"]

    shifts_unique = room_shifts["global_shift"].unique()

    total_penalty_all_shifts = 0

    print(
        f"Executando Metaheurística ({repetitions} repetições independentes por turno)..."
    )

    # for each unique shift (s)
    for s in shifts_unique:
        rooms_in_shift = room_shifts[room_shifts["global_shift"] == s]
        available_nurses = nurse_shifts[nurse_shifts["global_shift"] == s]

        if available_nurses.empty or rooms_in_shift.empty:
            continue

        env: NRA_Environment = NRA_Environment(rooms_in_shift, available_nurses, w_skill, w_work)  # type: ignore

        best_shift_fitness: float | int = float("inf")
        convergence_history: list = []

        # run "r" repetitions to analise
        for r in range(repetitions):
            ga = GeneticAlgorithm(
                env, pop_size=50, mutation_rate=0.05, generations=100, tournament_size=3
            )
            _, fitness = ga.run()

            if fitness < best_shift_fitness:
                best_shift_fitness = fitness
                # keep the best learn curve
                convergence_history = (
                    ga.history_best
                )

        total_penalty_all_shifts += best_shift_fitness

    print(f"Melhor Penalidade Total Encontrada (GA): {total_penalty_all_shifts}")
    return total_penalty_all_shifts